# Astropy y SunPy: de un archivo FITS a un mapa solar con coordenadas

**Escuela de Astrofísica Solar · HelioData Academy 2026 · GoSA – OAN, UNAL**

En este cuaderno recorremos el camino completo:

1. **Astropy** – abrir un archivo FITS, explorar su estructura (HDUs), leer el *header* (cartas: *keyword*, valor y comentario) y describir los datos (forma, tipo, mínimo, máximo, media, etc.).
2. **SunPy Map** – qué ventajas trae un `Map` frente a un arreglo de NumPy.
3. **Coordenadas helioproyectivas** – el punto (0, 0), el limbo oriental y el manejo de unidades.
4. **Coordenadas heliográficas** – Stonyhurst y Carrington, y transformaciones entre sistemas.
5. **Recorte (submapa)** – de píxeles a coordenadas del mundo y un recorte de 100″ × 100″.

**Datos:** SDO/HMI, 24 de octubre de 2014, cadencia de 45 s: continuo (`ic_45s`), magnetograma (`m_45s`) y dopplergrama (`v_45s`).

## 0. Preparación

Importamos las librerías que usaremos a lo largo de la clase.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Astropy: lectura de FITS, unidades, coordenadas y normalización de imágenes
from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import ImageNormalize

# SunPy: mapas y marcos de coordenadas solares
import sunpy
import sunpy.map
from sunpy.coordinates import frames

import astropy
print("Astropy:", astropy.__version__)
print("SunPy:  ", sunpy.__version__)

### 0.1 (Opcional) Descarga de los datos con Fido

Los archivos ya están en `../03_Datos/`. Esta celda solo se ejecuta si `DESCARGAR = True`; sirve para mostrar de dónde salieron.

In [ ]:
DESCARGAR = False   # cambiar a True solo si hace falta volver a descargar

download_dir = "../03_Datos/"

if DESCARGAR:
    from sunpy.net import Fido, attrs as a

    query_short = Fido.search(
        a.Time("2014-10-24 12:00:00", "2014-10-24 12:02:00"),
        a.Instrument.hmi,
        a.Physobs("intensity") | a.Physobs("LOS_magnetic_field") | a.Physobs("LOS_velocity"),
    )
    print(query_short)

    files = Fido.fetch(query_short, path=download_dir + "/{file}")
    print(files)

### 0.2 Localizar los archivos

Usamos `pathlib` para listar los FITS y elegimos los tres observables de la **misma hora** (12:00:45 TAI).

In [ ]:
data_dir = Path("../03_Datos/")

archivos = sorted(data_dir.glob("*.fits"))
for f in archivos:
    print(f.name)

# Tres observables tomados al mismo tiempo
tiempo = "20141024_120045"
f_cont = next(data_dir.glob(f"hmi.ic_45s.{tiempo}*continuum.fits"))
f_mag  = next(data_dir.glob(f"hmi.m_45s.{tiempo}*magnetogram.fits"))
f_dop  = next(data_dir.glob(f"hmi.v_45s.{tiempo}*Dopplergram.fits"))

print("\nContinuo:     ", f_cont.name)
print("Magnetograma: ", f_mag.name)
print("Dopplergrama: ", f_dop.name)

---
# Parte 1 · Astropy: abrir y explorar un FITS

## 1.1 Abrir el archivo y ver sus HDUs

`fits.open()` devuelve una **HDUList**: una lista de *Header Data Units* (HDU = header + data).
`hdul.info()` resume cuántas HDUs hay, su tipo y las dimensiones de los datos.

In [ ]:
hdul = fits.open(f_cont)
hdul.info()

**¿Por qué hay dos HDUs?** Los archivos de HMI que entrega JSOC vienen **comprimidos** (Rice):

- **HDU 0 (PRIMARY):** está vacía (no tiene datos).
- **HDU 1 (CompImageHDU):** contiene la imagen comprimida. Astropy la descomprime automáticamente al leerla.

Por eso no conviene suponer que los datos están siempre en `hdul[0]`. La siguiente función busca la primera HDU que sí tenga datos.

In [ ]:
def hdu_con_datos(hdul):
    '''Devuelve la primera HDU de la lista que contenga datos.'''
    for i, hdu in enumerate(hdul):
        if hdu.data is not None:
            print(f"Datos encontrados en la HDU {i} ({type(hdu).__name__})")
            return hdu
    raise ValueError("Ninguna HDU contiene datos")

hdu = hdu_con_datos(hdul)
header = hdu.header
data = hdu.data

## 1.2 El header: la ficha técnica de la observación

El header es una colección de **cartas** (*cards*) de 80 caracteres con la forma:

```
KEYWORD = valor / comentario
```

Primero veamos cuántas cartas hay y las primeras líneas tal como aparecen en el archivo.

In [ ]:
print("Número de cartas en el header:", len(header))
print()

# repr(header) muestra el header tal como está escrito en el archivo
print("\n".join(repr(header).splitlines()[:25]))

### Keyword, valor y comentario de una carta

Cada carta se puede leer de tres formas:

- `header["KEYWORD"]` → el **valor**
- `header.comments["KEYWORD"]` → el **comentario**
- `header.cards["KEYWORD"]` → la carta completa (con `.keyword`, `.value` y `.comment`)

In [ ]:
carta = header.cards["DATE-OBS"]

print("Keyword:   ", carta.keyword)
print("Valor:     ", carta.value)
print("Comentario:", carta.comment)
print()
print("Mismo resultado con acceso tipo diccionario:")
print(header["DATE-OBS"], "|", header.comments["DATE-OBS"])

### Cartas importantes, agrupadas por lo que describen

Recorremos un conjunto de keywords agrupadas por tema. Si alguna no existe en el archivo, lo indicamos en lugar de generar un error.

In [ ]:
grupos = {
    "Estructura":           ["NAXIS", "NAXIS1", "NAXIS2", "BITPIX"],
    "Tiempo":               ["DATE-OBS", "T_OBS", "EXPTIME"],
    "Instrumento":          ["TELESCOP", "INSTRUME", "WAVELNTH", "CONTENT"],
    "Unidades":             ["BUNIT"],
    "WCS (coordenadas)":    ["CTYPE1", "CTYPE2", "CUNIT1", "CUNIT2",
                             "CRPIX1", "CRPIX2", "CRVAL1", "CRVAL2",
                             "CDELT1", "CDELT2", "CROTA2"],
    "Sol y observador":     ["RSUN_OBS", "DSUN_OBS", "CRLN_OBS", "CRLT_OBS"],
}

for grupo, keywords in grupos.items():
    print(f"\n=== {grupo} ===")
    for key in keywords:
        if key in header:
            c = header.cards[key]
            print(f"{c.keyword:<10} = {str(c.value):<28} / {c.comment}")
        else:
            print(f"{key:<10}   (no está en este header)")

**Para comentar en clase:**

- `CTYPE1 = HPLN-TAN` y `CTYPE2 = HPLT-TAN` → las coordenadas son **helioproyectivas** (longitud y latitud en el cielo), con proyección tangente.
- `CRPIX` es el **píxel de referencia**, `CRVAL` la coordenada del mundo en ese píxel y `CDELT` la escala (″/píxel).
- `CROTA2 ≈ 180°` → las imágenes de HMI están giradas: el norte solar queda abajo si graficamos el arreglo "crudo".
- `RSUN_OBS` es el radio aparente del Sol en arcosegundos visto desde SDO: lo usaremos para ubicar el limbo.

## 1.3 Generalidades de los datos

`data` es un arreglo de NumPy. Revisamos forma, tipo y estadísticas.
Fuera del disco solar los píxeles valen **NaN**, por eso usamos las funciones `np.nan*`.

In [ ]:
def resumen_datos(data, nombre=""):
    '''Imprime un resumen estadístico ignorando los NaN.'''
    validos = np.isfinite(data)
    print(f"--- {nombre} ---")
    print(f"shape (ny, nx):       {data.shape}")
    print(f"ndim:                 {data.ndim}")
    print(f"dtype:                {data.dtype}")
    print(f"número de píxeles:    {data.size}")
    print(f"píxeles válidos:      {validos.sum()}  ({100 * validos.mean():.1f} %)")
    print(f"píxeles NaN:          {(~validos).sum()}")
    print(f"mínimo:               {np.nanmin(data):.3f}")
    print(f"máximo:               {np.nanmax(data):.3f}")
    print(f"media:                {np.nanmean(data):.3f}")
    print(f"mediana:              {np.nanmedian(data):.3f}")
    print(f"desviación estándar:  {np.nanstd(data):.3f}")
    p1, p99 = np.nanpercentile(data, [1, 99])
    print(f"percentiles 1 % / 99 %: {p1:.3f} / {p99:.3f}")

resumen_datos(data, f"Continuo  [{header.get('BUNIT', 'sin unidades')}]")

### Acceso a píxeles: ¡cuidado con el orden de los índices!

NumPy indexa como `data[fila, columna]`, es decir `data[y, x]`, y cuenta desde **0**.
DS9 y la convención FITS usan `(x, y)` y cuentan desde **1**.

In [ ]:
ny, nx = data.shape
yc, xc = ny // 2, nx // 2

print(f"Valor en el centro del arreglo data[{yc}, {xc}] = {data[yc, xc]:.2f}")
print(f"Valor en la esquina  data[0, 0] = {data[0, 0]}   (fuera del disco → NaN)")

### Histograma y visualización "cruda" con Matplotlib

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Histograma de los valores válidos
ax1.hist(data[np.isfinite(data)].ravel(), bins=200, color="goldenrod")
ax1.set_xlabel(f"Intensidad [{header.get('BUNIT', '')}]")
ax1.set_ylabel("Número de píxeles")
ax1.set_title("Distribución de intensidades")
ax1.set_yscale("log")

# Imagen: origin="lower" para que (0, 0) quede abajo a la izquierda, como en DS9
im = ax2.imshow(data, origin="lower", cmap="gray")
fig.colorbar(im, ax=ax2, label=header.get("BUNIT", ""))
ax2.set_title(f"Continuo HMI · {header['DATE-OBS']}")
ax2.set_xlabel("x [píxel]")
ax2.set_ylabel("y [píxel]")

plt.tight_layout()
plt.show()

**Observación:** los ejes están en **píxeles** y la imagen aparece invertida (por `CROTA2 ≈ 180°`).
Matplotlib no sabe nada del Sol: solo ve números. Esa es justamente la motivación para usar SunPy.

### Resumen de los tres observables

In [ ]:
for f in [f_cont, f_mag, f_dop]:
    with fits.open(f) as h:
        hd = hdu_con_datos(h)
        resumen_datos(hd.data, f"{hd.header.get('CONTENT', f.name)}  [{hd.header.get('BUNIT', '')}]")
        print()

hdul.close()   # cerramos el archivo que abrimos al comienzo

---
# Parte 2 · SunPy Map

Un `Map` junta en un solo objeto:

**Map = datos + metadata + WCS**

Ventajas frente a trabajar con el arreglo y el header por separado:

- Lee la metadata y la expone como **atributos con unidades** (`m.date`, `m.scale`, `m.rsun_obs`, …).
- Conoce su **sistema de coordenadas** y la **posición del observador**.
- Grafica con **ejes en coordenadas solares** (arcsec) y con la orientación correcta.
- Permite **transformar coordenadas**, hacer **recortes** en coordenadas físicas, rotar, reescalar, etc.
- Entiende instrumentos concretos (HMI, AIA, EIT, …) y aplica sus mapas de color.

In [ ]:
m_cont = sunpy.map.Map(f_cont)
m_mag  = sunpy.map.Map(f_mag)
m_dop  = sunpy.map.Map(f_dop)

m_cont   # en Jupyter muestra un resumen enriquecido del mapa

### Atributos útiles del mapa

Todo lo que antes buscábamos a mano en el header ahora está disponible directamente, y con unidades.

In [ ]:
m = m_cont   # trabajaremos con el continuo como mapa principal

print("Fecha:               ", m.date)
print("Observatorio:        ", m.observatory)
print("Instrumento:         ", m.instrument)
print("Medición:            ", m.measurement)
print("Longitud de onda:    ", m.wavelength)
print("Unidades:            ", m.unit)
print("Dimensiones:         ", m.dimensions)
print("Escala:              ", m.scale)
print("Píxel de referencia: ", m.reference_pixel)
print("Coordenada de ref.:  ", m.reference_coordinate)
print("Radio solar aparente:", m.rsun_obs)
print("Marco de coordenadas:", m.coordinate_frame)
print("Observador:          ", m.observer_coordinate)

### Graficar con coordenadas

Al crear el eje con `projection=m`, Matplotlib usa el WCS del mapa: los ejes quedan en arcsec (helioproyectivas) y el norte solar hacia arriba en la etiqueta de ejes.

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_limb(axes=ax, color="cyan")
plt.colorbar(label=str(m.unit))
plt.show()

### Los tres observables lado a lado

Ajustamos la normalización de cada mapa para ver mejor su contenido físico.

In [ ]:
# Magnetograma: escala simétrica en gauss
m_mag.plot_settings["norm"] = ImageNormalize(vmin=-1000, vmax=1000)

# Dopplergrama: escala simétrica según los percentiles de los datos
v = np.nanpercentile(np.abs(m_dop.data), 99)
m_dop.plot_settings["norm"] = ImageNormalize(vmin=-v, vmax=v)
m_dop.plot_settings["cmap"] = "RdBu_r"

fig = plt.figure(figsize=(18, 6))
for i, mapa in enumerate([m_cont, m_mag, m_dop]):
    ax = fig.add_subplot(1, 3, i + 1, projection=mapa)
    mapa.plot(axes=ax)
    ax.set_title(f"{mapa.measurement}\n{mapa.date.iso[:19]}")
    plt.colorbar(ax=ax, fraction=0.046, pad=0.04, label=str(mapa.unit))
plt.tight_layout()
plt.show()

> **Nota:** si se quiere el arreglo con el norte arriba (por ejemplo para exportarlo), se puede usar `m.rotate(order=3)`.
> Aquí **no** lo hacemos, porque en la Parte 5 usaremos coordenadas en píxeles del archivo original.

---
# Parte 3 · Coordenadas helioproyectivas

Según el header (`CTYPE = HPLN-TAN / HPLT-TAN`), los datos están en **coordenadas helioproyectivas** $(T_x, T_y)$:
ángulos en el cielo medidos desde el observador, con el origen en el **centro del disco solar**.

## 3.1 El punto (0″, 0″)

Creamos una coordenada con `SkyCoord` en el mismo marco del mapa (`m.coordinate_frame`) y la ubicamos en píxeles.

In [ ]:
centro = SkyCoord(0 * u.arcsec, 0 * u.arcsec, frame=m.coordinate_frame)

px_centro = m.world_to_pixel(centro)
print(centro)
print(f"\nEn píxeles del arreglo: x = {px_centro.x:.2f}, y = {px_centro.y:.2f}")
print(f"CRPIX del header (desde 1): x = {m.meta['crpix1']:.2f}, y = {m.meta['crpix2']:.2f}")

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_limb(axes=ax, color="cyan")

ax.plot_coord(centro, "o", color="red", markersize=10, label="(0″, 0″)")
ax.legend(loc="upper right")
plt.show()

## 3.2 El limbo oriental y el manejo de unidades

El Sol ocupa aproximadamente **0.5°** en el cielo: ese es su **diámetro** aparente.
Por lo tanto, el limbo está a unos **0.25°** del centro. El **este solar** queda a la **izquierda** (valores negativos de $T_x$).

Primero hacemos la conversión de unidades de forma explícita con Astropy.

In [ ]:
diametro_aparente = 0.5 * u.deg
radio_aprox = diametro_aparente / 2

print("Diámetro aparente:", diametro_aparente, "=", diametro_aparente.to(u.arcsec))
print("Radio aproximado: ", radio_aprox, "=", radio_aprox.to(u.arcsec))

In [ ]:
# Punto del limbo oriental a partir de la conversión explícita a arcsec
Tx_limbo = -radio_aprox.to(u.arcsec)
limbo_arcsec = SkyCoord(Tx_limbo, 0 * u.arcsec, frame=m.coordinate_frame)
print(limbo_arcsec)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_limb(axes=ax, color="cyan")
ax.plot_coord(centro, "o", color="red", markersize=8, label="(0″, 0″)")
ax.plot_coord(limbo_arcsec, "o", color="lime", markersize=10,
              label=f"Tx = {Tx_limbo:.0f}")
ax.legend(loc="upper right")
plt.show()

### Lo mismo, pero **sin convertir**: directamente en grados

`SkyCoord` acepta cualquier unidad angular. Astropy hace la conversión internamente, así que no hace falta pasar a arcsec a mano.

In [ ]:
limbo_grados = SkyCoord(-0.25 * u.deg, 0 * u.deg, frame=m.coordinate_frame)

print("Definido en grados:", limbo_grados.Tx)
print("Leído en arcsec:   ", limbo_grados.Tx.to(u.arcsec))
print("¿Mismo punto?      ", u.isclose(limbo_grados.Tx, limbo_arcsec.Tx))

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_limb(axes=ax, color="cyan")
ax.plot_coord(limbo_grados, "x", color="magenta", markersize=14, mew=3,
              label="definido en grados")
ax.plot_coord(limbo_arcsec, "o", color="lime", markersize=10, mfc="none", mew=2,
              label="definido en arcsec")
ax.legend(loc="upper right")
plt.show()

### ¿Qué tan buena es la aproximación de 0.25°?

El radio aparente real depende de la distancia Tierra–Sol en esa fecha y está en la metadata (`RSUN_OBS`, `m.rsun_obs`).
Comparemos el punto aproximado con el limbo exacto.

In [ ]:
limbo_exacto = SkyCoord(-m.rsun_obs, 0 * u.arcsec, frame=m.coordinate_frame)

print(f"Radio aproximado (0.25°): {radio_aprox.to(u.arcsec):.1f}")
print(f"Radio exacto (RSUN_OBS):  {m.rsun_obs:.1f}")
print(f"Diferencia:               {(m.rsun_obs - radio_aprox).to(u.arcsec):.1f}")

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_limb(axes=ax, color="cyan")
ax.plot_coord(limbo_arcsec, "o", color="lime", markersize=10, label="aprox. (0.25°)")
ax.plot_coord(limbo_exacto, "*", color="red", markersize=14, label="exacto (RSUN_OBS)")
ax.legend(loc="upper right")
plt.show()

**Mensaje:** con coordenadas helioproyectivas es difícil saber, "a ojo", qué valor de $T_x$ cae exactamente sobre el limbo:
depende de la distancia del observador. Esa es la motivación para las coordenadas **heliográficas**.

---
# Parte 4 · Coordenadas heliográficas

Las coordenadas heliográficas están **sobre la superficie solar** (longitud y latitud en grados), como la geografía en la Tierra.

## 4.1 Stonyhurst

- Longitud 0° en el **meridiano central visto desde la Tierra**.
- El sistema **no rota** con el Sol: está fijo respecto a la línea Sol–Tierra.
- El limbo oriental está, por construcción, cerca de **longitud −90°**, sin importar la distancia del observador.

In [ ]:
# Un punto en el limbo oriental, sobre el ecuador, en Stonyhurst
limbo_hgs = SkyCoord(-90 * u.deg, 0 * u.deg,
                     frame=frames.HeliographicStonyhurst, obstime=m.date)

# Algunos puntos adicionales para ilustrar
puntos_hgs = SkyCoord([-60, 0, 45] * u.deg, [20, 0, -30] * u.deg,
                      frame=frames.HeliographicStonyhurst, obstime=m.date)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_grid(axes=ax, grid_spacing=15 * u.deg, color="white", alpha=0.6)   # rejilla Stonyhurst
ax.plot_coord(limbo_hgs, "*", color="red", markersize=15, label="HGS (−90°, 0°)")
ax.plot_coord(puntos_hgs, "o", color="yellow", markersize=8, label="otros puntos HGS")
ax.legend(loc="upper right")
plt.show()

### Transformación Stonyhurst → helioproyectivas

Con `.transform_to()` pasamos al marco del mapa. SunPy usa el tiempo y la posición del observador guardados en la metadata.

In [ ]:
limbo_hpc = limbo_hgs.transform_to(m.coordinate_frame)

print("Stonyhurst:     ", limbo_hgs.lon.to(u.deg), limbo_hgs.lat.to(u.deg))
print("Helioproyectivas:", limbo_hpc.Tx.to(u.arcsec), limbo_hpc.Ty.to(u.arcsec))
print("RSUN_OBS:        ", m.rsun_obs)

### Transformación helioproyectivas → Stonyhurst

Las coordenadas helioproyectivas 2D no tienen distancia. SunPy supone que el punto está **sobre la superficie solar**
(si está fuera del disco, el resultado es NaN).

In [ ]:
centro_hgs = centro.transform_to(frames.HeliographicStonyhurst(obstime=m.date))
print("(0″, 0″) en Stonyhurst:", centro_hgs.lon.to(u.deg), centro_hgs.lat.to(u.deg))

print("\nLa latitud no es exactamente 0° por el ángulo B0 (inclinación del eje solar vista desde la Tierra).")
print("B0 del observador:", m.observer_coordinate.lat.to(u.deg))

## 4.2 Carrington

- El sistema **rota con el Sol** (período sinódico de ≈ 27.27 días).
- Una estructura en la superficie conserva (aproximadamente) su longitud Carrington mientras el Sol rota.
- Es el sistema natural para **seguir regiones activas** de una rotación a otra.

La longitud Carrington del meridiano central en esta observación está en el header (`CRLN_OBS`) y en el mapa.

In [ ]:
print("Longitud Carrington del centro del disco:", m.carrington_longitude)
print("CRLN_OBS del header:                     ", m.meta.get("crln_obs"))

In [ ]:
# Rejilla Carrington sobre el mapa
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_grid(axes=ax, grid_spacing=15 * u.deg, system="carrington",
            color="orange", alpha=0.7)
ax.plot_coord(limbo_hgs, "*", color="red", markersize=15, label="limbo oriental")
ax.legend(loc="upper right")
ax.set_title("Rejilla Carrington")
plt.show()

### Transformaciones Stonyhurst ↔ Carrington ↔ helioproyectivas

La **latitud** es la misma en ambos sistemas heliográficos; solo cambia la **longitud** (un desplazamiento que depende del tiempo).

In [ ]:
carrington = frames.HeliographicCarrington(observer=m.observer_coordinate, obstime=m.date)

# Stonyhurst → Carrington (mismos puntos que antes)
todos_hgs = SkyCoord([-90, -60, 0, 45] * u.deg, [0, 20, 0, -30] * u.deg,
                     frame=frames.HeliographicStonyhurst, obstime=m.date)
todos_hgc = todos_hgs.transform_to(carrington)
todos_hpc = todos_hgs.transform_to(m.coordinate_frame)

print(f"{'Stonyhurst (lon, lat)':>24} | {'Carrington (lon, lat)':>24} | {'Helioproy. (Tx, Ty)':>24}")
print("-" * 80)
for s, c, h in zip(todos_hgs, todos_hgc, todos_hpc):
    print(f"{s.lon.deg:9.2f}°, {s.lat.deg:7.2f}°    | "
          f"{c.lon.deg:9.2f}°, {c.lat.deg:7.2f}°    | "
          f"{h.Tx.to_value(u.arcsec):8.1f}″, {h.Ty.to_value(u.arcsec):8.1f}″")

In [ ]:
# Carrington → helioproyectivas: definimos un punto directamente en Carrington
lon_car = m.carrington_longitude - 30 * u.deg     # 30° al este del meridiano central
punto_hgc = SkyCoord(lon_car, 15 * u.deg, frame=carrington)

print("Carrington:      ", punto_hgc.lon.to(u.deg), punto_hgc.lat.to(u.deg))
print("Stonyhurst:      ", punto_hgc.transform_to(frames.HeliographicStonyhurst(obstime=m.date)).lon.to(u.deg))
print("Helioproyectivas:", punto_hgc.transform_to(m.coordinate_frame).Tx.to(u.arcsec),
      punto_hgc.transform_to(m.coordinate_frame).Ty.to(u.arcsec))

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_grid(axes=ax, grid_spacing=15 * u.deg, system="carrington", color="orange", alpha=0.5)
ax.plot_coord(punto_hgc, "o", color="red", markersize=10, label="punto Carrington")
ax.legend(loc="upper right")
plt.show()

---
# Parte 5 · Recorte del mapa (submapa)

1. Fijamos el **centro del recorte en píxeles**: (x, y) = (2838, 2595).
2. Lo transformamos a **coordenadas del mundo** con `pixel_to_world` (helioproyectivas, según el header).
3. Construimos un recuadro de **100″ × 100″** centrado en ese punto y hacemos el recorte con `submap`.

> En `pixel_to_world` los píxeles se cuentan desde **0** (convención de Python/NumPy).

In [ ]:
x_pix, y_pix = 2838, 2595

centro_recorte = m.pixel_to_world(x_pix * u.pix, y_pix * u.pix)
print("Centro del recorte:")
print(f"  píxeles:          ({x_pix}, {y_pix})")
print(f"  helioproyectivas: Tx = {centro_recorte.Tx.to(u.arcsec):.1f}, Ty = {centro_recorte.Ty.to(u.arcsec):.1f}")
print(f"  marco:            {centro_recorte.frame.name}")

# De paso: ¿dónde está este punto en coordenadas heliográficas?
c_hgs = centro_recorte.transform_to(frames.HeliographicStonyhurst(obstime=m.date))
c_hgc = centro_recorte.transform_to(carrington)
print(f"  Stonyhurst:       lon = {c_hgs.lon.deg:.2f}°, lat = {c_hgs.lat.deg:.2f}°")
print(f"  Carrington:       lon = {c_hgc.lon.deg:.2f}°, lat = {c_hgc.lat.deg:.2f}°")

In [ ]:
# Recuadro de 100" x 100" centrado en el punto
lado = 100 * u.arcsec

esquina_inf_izq = SkyCoord(centro_recorte.Tx - lado / 2, centro_recorte.Ty - lado / 2,
                           frame=m.coordinate_frame)
esquina_sup_der = SkyCoord(centro_recorte.Tx + lado / 2, centro_recorte.Ty + lado / 2,
                           frame=m.coordinate_frame)

# Mostramos el recuadro sobre el disco completo
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=m)
m.plot(axes=ax)
m.draw_quadrangle(esquina_inf_izq, top_right=esquina_sup_der, axes=ax,
                  edgecolor="red", linewidth=2)
ax.plot_coord(centro_recorte, "+", color="red", markersize=12)
plt.show()

In [ ]:
sub_cont = m.submap(esquina_inf_izq, top_right=esquina_sup_der)

print("Dimensiones del recorte:", sub_cont.dimensions)
print("Escala:                 ", sub_cont.scale)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection=sub_cont)
sub_cont.plot(axes=ax)
ax.plot_coord(centro_recorte, "+", color="red", markersize=15, mew=2)
plt.colorbar(label=str(sub_cont.unit))
plt.show()

### El mismo recorte en los tres observables

Como el recuadro está definido en **coordenadas del mundo** y no en píxeles, se puede aplicar directamente a los otros mapas.
Cada mapa usa su propio WCS, así que el recorte cae en el mismo lugar del Sol aunque el apuntado cambie ligeramente entre archivos.

In [ ]:
sub_mag = m_mag.submap(esquina_inf_izq, top_right=esquina_sup_der)
sub_dop = m_dop.submap(esquina_inf_izq, top_right=esquina_sup_der)

fig = plt.figure(figsize=(18, 6))
for i, sub in enumerate([sub_cont, sub_mag, sub_dop]):
    ax = fig.add_subplot(1, 3, i + 1, projection=sub)
    sub.plot(axes=ax)
    ax.plot_coord(centro_recorte, "+", color="black", markersize=12, mew=2)
    ax.set_title(sub.measurement)
    plt.colorbar(ax=ax, fraction=0.046, pad=0.04, label=str(sub.unit))
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas del recorte (ya no hay NaN porque estamos dentro del disco)
resumen_datos(sub_cont.data, "Recorte · continuo")
print()
resumen_datos(sub_mag.data, "Recorte · magnetograma")

---
## Resumen

| Paso | Herramienta | Idea |
|---|---|---|
| Abrir el archivo y ver HDUs | `fits.open`, `hdul.info()` | Un FITS es una lista de HDUs |
| Leer metadata | `header`, `header.cards` | Keyword = valor / comentario |
| Describir los datos | NumPy (`nanmean`, `shape`, …) | Los datos son un arreglo; fuera del disco hay NaN |
| Crear un mapa | `sunpy.map.Map` | Datos + metadata + WCS |
| Ubicar puntos | `SkyCoord`, `plot_coord` | Helioproyectivas en arcsec, con unidades |
| Cambiar de sistema | `.transform_to()` | Stonyhurst y Carrington sobre la superficie |
| Recortar | `pixel_to_world`, `submap` | Recortes definidos en coordenadas físicas |